In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))
    
    
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.social_media_collection import SocialMediaModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metric = ClassificationMetrics()

In [2]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

In [3]:
social_multi_paths = list((CONFIG['model_dir'] / 'binary' ).glob('social_media_*.joblib'))
scaler_multi_path = CONFIG['model_dir'] / 'helper' / 'social_media_max_abs_scaler.joblib'
nb_tfidf_multi_path = CONFIG['model_dir'] / 'helper' / 'social_media_nb_tfidf.joblib'

In [4]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"


# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

In [5]:
# create 3-class
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary',
    # data_source_column='data_source'
)
val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary',
    # data_source_column='data_source'
)

In [6]:
social_media_collection = SocialMediaModelCollection(
    model_joblibs=social_multi_paths,
    scaler_path=scaler_multi_path,
    nb_tfidf_path=nb_tfidf_multi_path,
)

ensemble_multi = Ensemble(model_collections=[social_media_collection])

In [7]:
social_media_collection.classifiers

{PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_ComplementNB_model.joblib'): Pipeline(steps=[('clf', ComplementNB(alpha=1.2006196407511314))]),
 PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LinearSVC_(Optuna)_model.joblib'): Pipeline(steps=[('clf',
                  LinearSVC(C=0.7813882258601701, class_weight='balanced',
                            dual=False, max_iter=5000, penalty='l1',
                            random_state=42))]),
 PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LogisticRegressionCV_model.joblib'): Pipeline(steps=[('clf',
                  LogisticRegressionCV(class_weight='balanced',
                                       cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                                       max_iter=1000, n_

In [9]:
# simple majority vote ensemble
train_pred = ensemble_multi.predict_simple_majority(train[tc])
metric.metrics(train['label_binary'], train_pred)

Predicting with SimpleMajority...
Input has 53707 samples.
social_media_ComplementNB_model: (53707,)
social_media_LinearSVC_(Optuna)_model: (53707,)
social_media_LogisticRegressionCV_model: (53707,)
Collected predictions from 3 models.
Prediction matrix shape: (53707, 3)
Aggregating predictions from 3 models...


{'CV Macro F1': 0.7280726618397022,
 'CV Weighted F1': 0.8043484642496677,
 'Accuracy': 0.8059470832479937,
 'Coverage': np.float64(1.0),
 'Precision': 0.5966524450278963,
 'FPR': np.float64(0.12011043514378558),
 'FNR': np.float64(0.43089685396775707),
 'TPR': np.float64(0.5691031460322429),
 'TNR': np.float64(0.8798895648562144)}

In [12]:
val_pred = ensemble_multi.predict_simple_majority(val[tc])
metric.metrics(val['label_binary'], val_pred)

Predicting with SimpleMajority...
Input has 17056 samples.
social_media_ComplementNB_model: (17056,)
social_media_LinearSVC_(Optuna)_model: (17056,)
social_media_LogisticRegressionCV_model: (17056,)
Collected predictions from 3 models.
Prediction matrix shape: (17056, 3)
Aggregating predictions from 3 models...


{'CV Macro F1': 0.6544739278805358,
 'CV Weighted F1': 0.7569689090364098,
 'Accuracy': 0.7495309568480301,
 'Coverage': np.float64(1.0),
 'Precision': 0.4356413166855846,
 'FPR': np.float64(0.1862032806531346),
 'FNR': np.float64(0.48205128205128206),
 'TPR': np.float64(0.517948717948718),
 'TNR': np.float64(0.8137967193468654)}